# Session 8: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over an alternative investments knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using the Alternative Investments Handbook - a comprehensive resource covering alternative investment strategies, reinsurance, private equity, real estate, commodities, and portfolio management.

### Data Preparation

We'll load the investments handbook as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/AlternativeInvestmentsHandbook.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
investment_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(investment_docs)}")
print(f"\nExample chunk:\n{investment_docs[0]}")

Raw documents: 1
Split chunks: 77

Example chunk:
page_content='The Alternative Investments Handbook
A Practical Guide to Non-Traditional Asset Classes and Risk Premiums

PART 1: FOUNDATIONS OF ALTERNATIVE INVESTING

Chapter 1: What Are Alternative Investments?' metadata={'source': 'data/AlternativeInvestmentsHandbook.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "investments_handbook".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    investment_docs,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include examining multiple critical aspects to ensure thorough evaluation. These steps are:\n\n1. **Investment Strategy and Process:** Assess whether the strategy is clearly defined, repeatable, and based on a solid theoretical foundation for generating returns.\n\n2. **Manager's Track Record:** Review the manager’s historical performance across different market cycles, and evaluate if their returns align with the stated strategy.\n\n3. **Risk Management:** Investigate what controls are in place to limit losses, how leverage is managed, and understand the worst-case scenarios.\n\n4. **Sources of Return:** Understand whether returns are driven by risk premiums, alpha, or structural advantages.\n\n5. **Liquidity and Redemption Terms:** Evaluate the liquidity profile, including lock-up periods, redemption notice requirements, and gate provisions to see if they are consistent with the investment strategy.\n\n6. **Fee Structure:** Ana

In [11]:
naive_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This low correlation stems from the fact that reinsurance risks are driven by natural catastrophes and weather events—such as hurricanes, earthquakes, and severe storms—rather than economic conditions. Additionally, reinsurance investments can offer attractive risk premiums, have short durations, and allow risk to be diversely spread across different geographies and peril types. These characteristics help reduce overall portfolio volatility and enhance diversification.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt financing, with the goal of improving operations, reducing costs, and growing revenue before exiting via sale or IPO.\n\n2. **Growth Equity:** Investing in established companies that need capital to expand, often for geographic expansion, product development, or acquisitions.\n\n3. **Distressed Investing:** Purchasing debt or equity of financially troubled companies at discounts, with value creation through financial restructuring or operational improvements.\n\n4. **Secondaries:** Buying existing private equity fund interests from other investors, offering diversification across different vintage years and strategies.\n\n5. **Event-Driven Strategies:** Investing around corporate events such as mergers, acquisitions, restructurings, or bankruptcies, including merger arbitrage.\n\nThese strategies aim to generate returns through

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(investment_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include:\n\n1. **Evaluate the Investment Strategy and Process:** Ensure the strategy is clearly defined, repeatable, and based on a sound theoretical basis for generating returns.\n\n2. **Assess the Manager's Track Record:** Review past performance across multiple market cycles to determine if returns are consistent with the stated strategy.\n\n3. **Examine Risk Management Measures:** Understand the controls in place to limit losses, how leverage is managed, and consider worst-case scenarios.\n\n4. **Analyze the Source of Returns:** Determine whether returns come from risk premiums, alpha, or structural advantages.\n\n5. **Assess Liquidity and Redemption Terms:** Understand how liquid the investment is and the conditions for redemptions.\n\n6. **Review Fee Structure and Alignment:** Check how fees are structured and whether they align with performance incentives.\n\n7. **Evaluate Risk Management Systems:** Ensure appropriate syst

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

"Reinsurance provides portfolio diversification benefits because it involves assets or investments, such as catastrophe bonds and collateralized reinsurance, that are generally lowly correlated with traditional financial markets. For example, catastrophe bonds (a common form of Insurance-Linked Securities) deliver returns tied to natural disasters, which are independent of stock or bond market movements. During periods of financial turmoil, these assets often maintain neutral or positive returns, reducing overall portfolio volatility. This low correlation helps spread risk and improve the portfolio's risk-adjusted returns by protecting it from systemic market downturns."

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing typically include buyouts, venture capital, and growth equity. These strategies involve investing in private companies or taking private positions in publicly traded companies to generate returns through operational improvements, strategic growth, or eventual exit via sale or IPO.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

The BM25 retriever may be more effective for answering queries that require an exact keyword match from some source document, for example:<br><br>
"What is the name of Stone Ridge's Flagship Reinsurance Mutual Fund?"<br><br>
Since the retrieval algorithm tracks which specific words each document section contains, and not necessarily word order, it is good at determining if a given word is present in a document, but may not be as good in extracting complex syntax or ideas from that document.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
# from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. Examining the investment strategy, including the source of returns such as risk premium, alpha, or structural advantages.\n2. Assessing the liquidity of the investment and understanding redemption terms.\n3. Analyzing the fee structure and how fees are aligned with performance.\n4. Evaluating the manager’s track record across different market environments.\n5. Reviewing the risk management systems in place.\n6. Considering how the investment fits within the overall portfolio.\n7. Understanding the tax implications associated with the investment.\n\nThese steps are critical because alternative investments tend to be less transparent and more complex than traditional investments.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits because its returns are driven by natural catastrophes and weather events, which have near-zero correlation with traditional financial markets like equities and bonds. This means that reinsurance-related investments tend to behave independently of economic conditions, helping to reduce overall portfolio risk. Additionally, since reinsurance risk can be spread across different geographies and peril types, it further enhances diversification. This unique risk profile makes reinsurance an attractive complement to other asset classes, offering potential risk premiums with low correlation to standard financial markets.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and debt financing with the goal of improving operations, reducing costs, and increasing revenue before exiting through a sale or IPO.\n\n2. **Direct Investments:** Investing directly in private companies or taking public companies private, with an emphasis on improving management, governance, and growth prospects to achieve profitable exits.\n\n3. **Event-Driven Strategies:** Investing around corporate events such as mergers, acquisitions, restructurings, and bankruptcies. A common example is merger arbitrage, which involves buying the target company and shorting the acquirer in announced deals.\n\nThese strategies focus on actively improving and leveraging various corporate events to generate returns.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!


In [25]:
# from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include examining several critical areas:\n\n1. **Investment Strategy and Process**:\n   - Assess whether the strategy is clearly defined and repeatable.\n   - Understand the theoretical basis for generating returns and whether it is sustainable.\n\n2. **Managertrack Record**:\n   - Evaluate the manager’s performance over multiple market cycles.\n   - Confirm that past returns are consistent with the stated strategy and suitable for your risk appetite.\n\n3. **Risk Management**:\n   - Review the risk management systems in place.\n   - Understand how losses are limited, leverage is managed, and what the worst-case scenarios could be.\n\n4. **Operational Infrastructure**:\n   - Ensure there is adequate separation of duties between portfolio management and back-office operations.\n   - Verify independent service providers for custody, administration, and auditing.\n\n5. **Legal and Regulatory Considerations**:\n   - Confirm the fund

In [28]:
multi_query_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This low correlation arises because reinsurance risks are driven by natural catastrophes such as hurricanes, earthquakes, and severe storms, rather than economic conditions. As a result, incorporating reinsurance investments—such as catastrophe bonds or collateralized reinsurance—into a portfolio can reduce overall volatility and improve risk-adjusted returns by adding a source of return that behaves independently of traditional assets.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and significant debt financing. The goal is to improve operations, reduce costs, and grow revenue before selling or taking the company public.\n\n2. **Growth Equity:** Investing in established, profitable companies that require capital to expand—such as geographic growth, product development, or acquisitions.\n\n3. **Venture Capital:** Focusing on early-stage, high-growth companies, especially in sectors like technology and healthcare.\n\n4. **Distressed Investing:** Purchasing debt or equity of financially troubled companies at significant discounts, with value added through restructuring, operational improvements, or asset sales.\n\n5. **Secondaries:** Buying existing private equity interests from other investors, offering diversification across vintage years and strategies, often at a discount.\n\n6. **Real Estate and Real Assets:** 

### Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:
Generating multiple reformulations can improve recall by growing the search space of vectors that the LLM uses to find its answer. By re-wording the question in subsequent formulations you can potentially add semantically similar vectors to the answer search space that may not have been included in prior versions of the question, based on the vocabulary, framing or level of detail used in the subsequent questions.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [32]:
# from langchain.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers import ParentDocumentRetriever
# from langchain.storage import InMemoryStore
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [34]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="investments_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="investments_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [35]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [36]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [37]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [38]:
parent_document_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include:\n\n1. **Assessing the Investment Strategy and Process**:\n   - Ensure the strategy is clearly defined, repeatable, and based on a sound theoretical rationale for generating returns.\n\n2. **Evaluating Track Record**:\n   - Review the manager’s historical performance across multiple market cycles to determine consistency and alignment with the stated strategy.\n\n3. **Analyzing Risk Management Systems**:\n   - Understand what controls are in place to limit losses, how leverage is managed, and the scenarios considered for worst-case outcomes.\n\n4. **Reviewing Operational Infrastructure**:\n   - Confirm adequate separation of duties between portfolio management and back-office operations.\n   - Check for independent service providers for custody, administration, and auditing functions.\n\n5. **Examining Fee Structure**:\n   - Ensure the fees are reasonable relative to the value provided.\n   - Verify that fee arrangements 

In [39]:
parent_document_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits because its returns are largely uncorrelated with traditional financial markets like equities and bonds. This is because reinsurance risks are driven by natural catastrophe events such as hurricanes, earthquakes, floods, and wildfires, rather than economic or market conditions. As a result, investments in reinsurance or insurance-linked securities tend to be less affected by economic downturns, making them valuable for diversifying an investment portfolio. Additionally, reinsurance offers attractive risk premiums, short durations for regular income, and opportunities to spread risk across different geographies and peril types.'

In [40]:
parent_document_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. **Leveraged Buyouts (LBOs):** Acquiring mature companies using a combination of equity and significant debt. The objective is to improve operational efficiency, reduce costs, and grow revenue before eventually exiting through a sale or IPO. Historically, LBOs have yielded returns of 2-3 times the invested capital.\n\n2. **Growth Equity:** Investing in established, profitable companies that require capital for expansion activities such as geographic growth, product development, or acquisitions.\n\n3. **Distressed Investing:** Purchasing debt or equity of financially troubled companies at substantial discounts, with value creation through financial restructuring, operational improvements, or asset sales.\n\n4. **Secondaries:** Buying existing private equity fund interests from other investors, offering diversification across vintage years and strategies, often at a discount to net asset value.\n\nThese strategies collective

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
# from langchain.retrievers import EnsembleRetriever
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [43]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [44]:
ensemble_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

"The key due diligence steps for alternative investments include:\n\n1. **Examine Investment Strategy and Process:** Ensure the strategy is clearly defined, repeatable, and has a sound theoretical basis for generating returns.\n\n2. **Assess Manager Track Record:** Review the manager's performance across multiple market cycles to verify consistency with the stated strategy.\n\n3. **Evaluate Risk Management Systems:** Determine what controls are in place to limit losses, how leverage is managed, and the preparedness for worst-case scenarios.\n\n4. **Review Operational Infrastructure:** Check for adequate separation of duties, independence of service providers such as custody, administration, and auditing.\n\n5. **Analyze Fee Structure:** Confirm that fees are reasonable, properly aligned with performance, and include provisions like high-water marks or clawbacks to protect investors.\n\n6. **Assess Liquidity Terms:** Understand lock-up periods, redemption notice requirements, and gate p

In [45]:
ensemble_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are largely uncorrelated with traditional financial markets such as equities and bonds. This uncorrelated nature arises because the risks involved—natural catastrophes like hurricanes, earthquakes, and severe storms—are driven by weather and seismic events rather than economic factors. As a result, reinsurance investments can help reduce overall portfolio volatility and enhance risk-adjusted returns by adding exposure to these unique, geographically and peril-diversified risks that do not typically move in tandem with broader financial markets.'

In [46]:
ensemble_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of equity and debt financing with the goal of improving operations, reducing costs, and expanding revenue before exiting through a sale or IPO. LBOs have historically generated returns of 2-3x invested capital.\n\n2. Growth Equity: Investing in established companies that need capital to expand, often for geographic growth, new product development, or acquisitions.\n\n3. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, aiming to create value through financial restructuring, operational improvements, or asset sales.\n\n4. Secondaries: Buying existing private equity fund interests from other investors, providing diversification across vintage years and strategies, often at a discount to net asset value.\n\nAdditionally, private equity strategies may involve venture capital investments in early-s

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [48]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [49]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="investments_handbook_semantic_chunks"
)

We'll use naive retrieval for this example.

In [50]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [51]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [52]:
semantic_retrieval_chain.invoke({"question" : "What are the key due diligence steps for alternative investments?"})["response"].content

'The key due diligence steps for alternative investments include examining several critical areas to ensure thorough evaluation:\n\n1. Investment Strategy and Process:\n   - Is the strategy clearly defined and repeatable?\n   - What is the theoretical basis for generating returns?\n2. Track Record:\n   - How has the manager performed over multiple market cycles?\n   - Are historical returns consistent with the stated strategy?\n3. Risk Management:\n   - What controls are in place to limit losses?\n   - How is leverage managed?\n   - What are the worst-case scenarios?\n4. Operational Infrastructure:\n   - Is there adequate separation of duties between portfolio management and back-office operations?\n   - Are independent service providers involved in custody, administration, and auditing?\n5. Fee Structure:\n   - Are fees reasonable relative to the value added?\n   - Is the fee structure aligned with investor interests through mechanisms like high-water marks and clawback provisions?\n6

In [53]:
semantic_retrieval_chain.invoke({"question" : "How does reinsurance provide portfolio diversification benefits?"})["response"].content

'Reinsurance provides portfolio diversification benefits primarily because its returns are driven by natural catastrophe risks—such as hurricanes, earthquakes, floods, and wildfires—rather than by economic or financial market conditions. This means that reinsurance investments tend to have very low or near-zero correlation with traditional financial assets like stocks and bonds. \n\nSpecifically, the context highlights that reinsurance and insurance-linked securities (ILS) have returns that are largely uncorrelated with equity and bond markets because their performance depends on natural events rather than economic factors. For example, returns in catastrophe bonds are triggered by weather or seismic events, which are unpredictable and independent of financial market movements. Additionally, the correlation matrix reference indicates that reinsurance and catastrophe bonds have correlations close to zero or very low with equity markets.\n\nThis low correlation allows reinsurance to serv

In [54]:
semantic_retrieval_chain.invoke({"question" : "What are the main strategies for private equity investing?"})["response"].content

'The main strategies for private equity investing include:\n\n1. Leveraged Buyouts (LBOs): Acquiring mature companies using a combination of debt and equity, aiming to improve operations, reduce costs, and grow revenue before exiting through a sale or IPO. Historically, LBOs have generated returns of 2-3 times the invested capital.\n\n2. Growth Equity: Investing in established companies that require capital to expand their operations, market share, or product offerings.\n\n3. Venture Capital: Providing early-stage funding to high-growth-potential startups, particularly in sectors like technology and healthcare, often in stages such as seed, Series A, and subsequent rounds.\n\n4. Distressed Investing: Purchasing debt or equity of financially troubled companies at significant discounts, with value creation through restructuring and operational improvements.\n\n5. Secondaries: Buying existing private equity interests from other investors, offering diversification across vintages and strat

### Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

Sentences like FAQs are already naturally chunked, and many share similar vocabulary even though they cover different topics. This may result in chunks that span multiple Q&A boundaries, inadvertently mixing unrelated questions and responses together — failing to respect the natural Q&A structure and producing embedding vectors that are too similar to one another.<br><br>
To address this unintentional merging of unrelated content, you could raise the percentile threshold, requiring sentences to be even more semantically related before being grouped together. You could also use the gradient thresholding method, which splits chunks based on the rate of change in similarity between consecutive sentences rather than comparing against a global document-level threshold, which would better chunk on gradual topic differences that a static percentile threshold might miss. 

---

# Breakout Room Part #2

### Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [62]:
import nest_asyncio
nest_asyncio.apply()

import copy
import time
import pandas as pd
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)

# ── 1. Wrap LLM & Embeddings for Ragas ──────────────────────────────────────
# Use gpt-4.1-mini for generation — gpt-4.1-nano produces invalid JSON escapes
# (e.g. S\&P 500) that fail Ragas' ThemesExtractor output parser.
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# ── 2. Generate Synthetic Test Set ──────────────────────────────────────────
# Use generate_with_langchain_docs with raw_docs so Ragas handles its own
# internal chunking. This avoids both "too short" (chunked docs < 100 tokens)
# and "no relationships" (single raw doc = 1 node) errors.
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
testset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)
print(f"Generated {len(testset)} test samples")
testset.to_pandas()

# ── 3. Define the retrieval chains to evaluate ──────────────────────────────
retrieval_chains = {
    "Naive": naive_retrieval_chain,
    "BM25": bm25_retrieval_chain,
    "Contextual Compression (Rerank)": contextual_compression_retrieval_chain,
    "Multi-Query": multi_query_retrieval_chain,
    "Parent Document": parent_document_retrieval_chain,
    "Ensemble": ensemble_retrieval_chain,
    "Semantic Chunking": semantic_retrieval_chain,
}

# ── 4. Run each chain on the test set & evaluate ────────────────────────────
metrics = [
    LLMContextRecall(),
    ContextEntityRecall(),
    Faithfulness(),
    ResponseRelevancy(),
    FactualCorrectness(),
    NoiseSensitivity(),
]
custom_run_config = RunConfig(timeout=360)
all_results = {}

for name, chain in retrieval_chains.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    # Deep-copy the testset so each retriever gets its own clean copy
    ts_copy = copy.deepcopy(testset)

    # Populate responses & retrieved contexts
    start = time.time()
    for row in ts_copy:
        result = chain.invoke({"question": row.eval_sample.user_input})
        row.eval_sample.response = result["response"].content
        row.eval_sample.retrieved_contexts = [
            ctx.page_content for ctx in result["context"]
        ]
    latency = time.time() - start

    # Convert to pandas and keep only columns needed for evaluation
    # (drops persona_name, query_style, query_length which may be NaN
    #  and cause Pydantic validation errors)
    df_eval = ts_copy.to_pandas()
    essential = {"user_input", "response", "retrieved_contexts", "reference", "reference_contexts"}
    df_eval = df_eval[[c for c in df_eval.columns if c in essential]]
    eval_dataset = EvaluationDataset.from_pandas(df_eval)

    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=custom_run_config,
    )

    scores = result.to_pandas()[
        [c for c in result.to_pandas().columns if c not in ("user_input", "response", "retrieved_contexts", "reference")]
    ].mean()
    scores["avg_latency_per_query_s"] = latency / len(ts_copy)
    all_results[name] = scores
    print(f"  Latency: {latency:.1f}s total, {latency/len(ts_copy):.2f}s/query")
    print(scores.to_string())

# ── 5. Compile comparison table ─────────────────────────────────────────────
comparison_df = pd.DataFrame(all_results).T
comparison_df.index.name = "Retriever"
print("\n\n" + "="*80)
print("RETRIEVER COMPARISON")
print("="*80)
print(comparison_df.to_string())

# ── 6. Analysis ─────────────────────────────────────────────────────────────
print("""
================================================================================
ANALYSIS
================================================================================
The comparison above evaluates each retriever across six Ragas metrics plus
average latency per query.

Key considerations:
- COST: BM25 and Naive retrieval are the cheapest — they require no additional
  LLM or API calls beyond the single generation call. Multi-Query is the most
  expensive because it makes extra LLM calls to reformulate queries. Contextual
  Compression (Rerank) adds a Cohere API call per query. Ensemble multiplies
  the cost of its constituent retrievers.

- LATENCY: BM25 is typically the fastest (no embedding lookup). Naive is next.
  Multi-Query and Ensemble are the slowest due to multiple retrieval passes.
  Reranking adds moderate latency from the Cohere API call.

- PERFORMANCE: Contextual Compression (Rerank) and Ensemble tend to score
  highest on context recall and faithfulness because they either rerank for
  relevance or combine multiple retrieval signals. Parent Document retrieval
  excels when questions require broader context. BM25 can outperform dense
  retrieval on keyword-heavy queries but usually lags on semantic questions.

Refer to the comparison_df table above and your LangSmith traces for the
precise numbers on your run.
""")

/var/folders/12/x_26y7_51qz7j6vbvgjhrh580000gq/T/ipykernel_12325/2244023906.py:11: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
/var/folders/12/x_26y7_51qz7j6vbvgjhrh580000gq/T/ipykernel_12325/2244023906.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/var/folders/12/x_26y7_51qz7j6vbvgjhrh580000gq/T/ipykernel_12325/2244023906.py:11: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics impor

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

...

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

Generated 9 test samples

Evaluating: Naive


Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[35]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[53]: TimeoutError()


TypeError: Could not convert [list(['The Alternative Investments Handbook\nA Practical Guide to Non-Traditional Asset Classes and Risk Premiums\n\n PART 1: FOUNDATIONS OF ALTERNATIVE INVESTING\n\n Chapter 1: What Are Alternative Investments?\n\nAlternative investments encompass asset classes and strategies beyond traditional stocks and bonds. They include real estate, private equity, hedge funds, commodities, infrastructure, reinsurance, and other specialized strategies. Alternatives have grown significantly in institutional portfolios over the past two decades, with many pension funds, endowments, and sovereign wealth funds allocating 20-40% of their portfolios to alternative strategies.\n\nThe appeal of alternative investments lies in their ability to provide diversification, access to unique risk premiums, and potentially higher risk-adjusted returns. Because many alternatives have low correlation with traditional equity and bond markets, adding them to a portfolio can reduce overal